In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("../data/raw_games.csv")

In [3]:
# True Shooting % (Factors in 2PT, 3PT, and FTs)
df['TS_PCT'] = df['PTS'] / (2 * (df['FGA'] + 0.44 * df['FTA']))
# Effective Field Goal % (Adjusts for 3PT being worth more)
df['eFG_PCT'] = (df['FGM'] + 0.5 * df['FG3M']) / df['FGA']
# Fill any weird NaN divisions with 0
df['TS_PCT'] = df['TS_PCT'].fillna(0)
df['eFG_PCT'] = df['eFG_PCT'].fillna(0)

# 1. Assist to Turnover Ratio (Protect against 0 turnovers)
df['AST_TOV_RATIO'] = round(df['AST'] / df['TOV'].replace(0, 1), 2)

# 2. Calculate 2-Pointers Made (Total FGM - 3-Pointers Made)
fg2m = df['FGM'] - df['FG3M']

# 3. Point Distribution Percentages (Multiply by point values!)
df['PCT_PTS_2PT'] = round(((fg2m * 2) / df['PTS'].replace(0, 1)) * 100, 2)
df['PCT_PTS_3PT'] = round(((df['FG3M'] * 3) / df['PTS'].replace(0, 1)) * 100, 2)
df['PCT_PTS_FT']  = round((df['FTM'] / df['PTS'].replace(0, 1)) * 100, 2)


In [4]:
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
df = df.sort_values(by=['TEAM_ABBREVIATION', 'GAME_DATE'],ignore_index=True)
df

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,PTS,PLUS_MINUS,VIDEO_AVAILABLE,Season,TS_PCT,eFG_PCT,AST_TOV_RATIO,PCT_PTS_2PT,PCT_PTS_3PT,PCT_PTS_FT
0,22021,1610612737,ATL,Atlanta Hawks,22100014,2021-10-21,ATL vs. DAL,W,240,45,...,113,26,1,2021-22,0.576766,0.558511,2.38,53.10,39.82,7.08
1,22021,1610612737,ATL,Atlanta Hawks,22100027,2021-10-23,ATL @ CLE,L,240,38,...,95,-6,1,2021-22,0.449811,0.434343,2.22,58.95,31.58,9.47
2,22021,1610612737,ATL,Atlanta Hawks,22100043,2021-10-25,ATL vs. DET,W,240,46,...,122,18,1,2021-22,0.614672,0.577778,1.50,55.74,29.51,14.75
3,22021,1610612737,ATL,Atlanta Hawks,22100059,2021-10-27,ATL @ NOP,W,240,40,...,102,3,1,2021-22,0.492849,0.458333,1.91,62.75,23.53,13.73
4,22021,1610612737,ATL,Atlanta Hawks,22100066,2021-10-28,ATL @ WAS,L,240,48,...,111,-11,1,2021-22,0.589422,0.579545,2.00,75.68,16.22,8.11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12295,22025,1610612764,WAS,Washington Wizards,22501133,2026-04-05,WAS @ BKN,L,240,44,...,115,-6,1,2025-26,0.629654,0.613924,0.87,60.87,23.48,15.65
12296,22025,1610612764,WAS,Washington Wizards,22501148,2026-04-07,WAS vs. CHI,L,240,37,...,98,-31,1,2025-26,0.514490,0.494186,1.00,53.06,33.67,13.27
12297,22025,1610612764,WAS,Washington Wizards,22501166,2026-04-09,WAS vs. CHI,L,240,38,...,108,-11,1,2025-26,0.500556,0.447917,1.26,51.85,27.78,20.37
12298,22025,1610612764,WAS,Washington Wizards,22501172,2026-04-10,WAS vs. MIA,L,240,46,...,117,-23,1,2025-26,0.595966,0.576087,1.86,54.70,35.90,9.40


In [5]:
# Calculate the difference in days between the current game and their last game
df['REST_DAYS'] = df.groupby('TEAM_ABBREVIATION')['GAME_DATE'].diff().dt.days
# For the first game of the season (which will be NaN), assume 3+ days of rest
df['REST_DAYS'] = df['REST_DAYS'].fillna(3)

# Back-to-Back Flag: If Rest Days is exactly 1, they played yesterday (Massive Fatigue)
df['B2B'] = (df['REST_DAYS'] == 1).astype(int)

# Sort back to standard chronological order
df = df.sort_values(by='GAME_DATE').reset_index(drop=True)

In [6]:
# 3. Save the engineered base dataset
df.to_csv("../data/engineered_raw_games.csv", index=False)
print(f"Engineered features added! New shape: {df.shape}")
df[['TEAM_ABBREVIATION', 'GAME_DATE', 'TS_PCT', 'REST_DAYS', 'B2B']].head(10)

Engineered features added! New shape: (12300, 38)


,TEAM_ABBREVIATION,GAME_DATE,TS_PCT,REST_DAYS,B2B
0,GSW,2021-10-19,0.569680,3.0,0
1,LAL,2021-10-19,0.551471,3.0,0
2,BKN,2021-10-19,0.552486,3.0,0
3,MIL,2021-10-19,0.562345,3.0,0
4,CLE,2021-10-20,0.599485,3.0,0
5,CHI,2021-10-20,0.507559,3.0,0
6,OKC,2021-10-20,0.434695,3.0,0
7,BOS,2021-10-20,0.527061,3.0,0
8,MEM,2021-10-20,0.626900,3.0,0
9,TOR,2021-10-20,0.389014,3.0,0
